Broadcasting: The term broadcasting describes how Numpy treats arrays with different shapes during arithmetic operations. They are subjected to certain constraints, the smaller array is "broadcast" across the larger array so that they have compatible shapes.

Broadcasting - provides a means of vectorising array opeartions so that looping - occurs in C rather than in Python. Does without making a copy of data and leads to effiecient algorithm implementations.

There are however some cases where Broadcasting is a bad idea as it leads to ineffiecient use of memory that slows computation.
















In numpy we do operations on pair of arrays on an element-by-element basis.

simpleast case: array have same shap but the broadcasting rules are used when the arrays shape meets certain constraints.
Simplest broadcasting == when an array and a scaler values are combined in an operation. Here we can think it as:

Imagine that the scaler b is being stretched during an arithmetic operation into an array with the same shape as a. The new elements are simply copies of the scalar b.

The stretching is conceptual only! Numpy is so smart that it used the og scalar value without actually making copies so that: Broadcasting operation == More memory and computationally efficient as possible !

GENERAL BROADCASTING RULE:
When operating two arrays. Numpy compares their shape element wise. Starts with trailing rightmost dimension and works its way left.

Two D's are only compatible when:
1. they are equal in shape
2. one of them is 1.

And this is looks so easy but Numpy sometimes creates error while broadcasting.

If these conditions are not met a `ValueError: oparands could not be broadcast together` exception is thrown: indicates array shapes are incompatible.

Input arrays do not need to have the same number of dimensions.

The resulting array will have the same number of dimensions as the input array with the greatest number of dimensions, where the size of each dimension is the largest size of the corresponding dimension among the input array.

R_Array == same no. of dimensions as input array with greatest no. of dimension:
size fo each dimension == largest size of corresponding dimension among the input array.

When either of the dimensions compared is one, the other is used. In other words, dimensions with size 1 are stretched or “copied” to match the other.

In [ ]:
import numpy as np

In [ ]:
# Trap 1: (100,3) matrix. Subtract column means of shape (3,). Then try means.reshape(1,3) vs means.reshape(3,1). Which is correct? Verify by checking zero mean per column.

array = np.linspace(1, 100, 300).reshape((100,3))
column_mean = array.mean(axis=0,keepdims = True)
# a pro data scientist uses keepdims = True so that the array maintains itself.
# to enforce broadcasting i must reshape the array into a 2d one
# column_mean = column_mean.reshape((3,1))
print(column_mean)
print((array-column_mean).mean(axis=0))

What I predicted: This is my first tim analysing the broadcasting. At first i was just not sure and  predicted 3,1 as the answer

What really Happened,the correct version: I realised that 1 is prepended to match the shape of the other array: (3,) is shaped to (1,3) to match (100,3) and now the 100 scales the 1 and 3 is already matched with the 3 in other array so it becomes -> (100,3)

How would this silently corrupt the model: It would have lead to TypeError where the broadcasting would not be possible on the operands. and sometimes it silently corrupt by increasing the size of the array randomly

In [ ]:
# Trap 2: Two arrays (5,) and (5,). np.dot gives scalar. Reshape to (5,1) and (1,5). What does @ give now? Predict first.

# Prediction: @ will give (5,1) @ (1,5 )==( 5,5)
# but i do think that in numpy np.dot and @ follows same matrix multiplication i can be wrong and the answer still might be a scalar but we have reshaped it meaning yahhh ! It will be 5,5
array1 = np.arange(5)
array2 = np.arange(5)

print(np.dot(array1, array2))

array1 = array1.reshape(5,1)
array2 = array2.reshape(1,5)

print(array1 @ array2, (array1@array2).shape)

# prediction was right

What I predicted: I have worked on how `@` works when multiplying two arrays, and also worked with `np.dot()` method so I know for two 1D arrays it produces a scalar. I predicted a scalar and then an array of  (5,5) shape when (5,1) is multiplied with (1,5).

What really Happened,the correct version: I was correct and hence proved that broadcasting really stretches the 1 to match the corresponding shape!

How would this silently corrupt the model: Sometimes when we multiple an array with a scalar the array rather than scaling it gets scaled and bigger in size too. Much bigger than we initally wanted ! (there is also one more thing i am forgetting at this point)

In [ ]:
# Trap 3: Add bias (3,) to batch outputs (32,3). Try it. Now try shape (3,1) instead. What shape did you actually add?
import numpy as np
array1 = np.linspace(1,10, 96).reshape((32,3))
array2 = np.arange(3)

# Prediction: If i add 3, to 32,3 i will still have 32,3
# and if i reshaped 3, to 3,1 and then add to 32,1 i will get 32,32

#Let's see if my predictions are actually correct.

sum1 = array1 + array2 # here (3,) is treated by numpy as (1,3) to match the 2 dimn of (32,3)
print(sum1[0:5,:])
# sum2 = array1 + array2.reshape((3,1))

# Shit i was wrong ! Lesson Learned: It leads to value error as (32,3) and (3,1)
# 3 and 1 are rightmst so 1 can stretch to make itself 3, ut when we come to 100 and 3 100 != 3 and none of them is one and here is where the broadcating failed

What I predicted: My predictions were that if i add (3,1) to (32,1) i will get (32,32)

What really Happened,the correct version: I was wrong, I learned that 32 and 3 are not compatible and they are not able to broadcast together, later i studied and understood that (3,) in numpy gets prepended by 1 and becomes (1,3).

How would this silently corrupt the model: I would've let that happen and would've came around the TypeError! Also i qwas unknowingly making the array (32,32)

To verify the broadcasting behavior and guard against 'silent failures', it's good practice to explicitly print the shapes and inspect a few elements of the result. In the case of `array1 + array2`, the 1D array `array2` is broadcast across the rows of `array1`.

In [ ]:
print('Original shape of array1:', array1.shape)
print('Original shape of array2:', array2.shape)

# Perform the addition
sum_result = array1 + array2

print('Shape of the result (sum_result):', sum_result.shape)

print('\nFirst 5 rows of array1:\n', array1[0:5,:])
print('\narray2 (broadcast across rows):\n', array2)
print('\nFirst 5 rows of sum_result (should be array1[i,:] + array2):\n', sum_result[0:5,:])

# Verification for a specific element (e.g., first row, first column)
# sum_result[0, 0] should be array1[0, 0] + array2[0]
expected_value_0_0 = array1[0, 0] + array2[0]
print(f'\nVerifying sum_result[0, 0]: Expected {expected_value_0_0}, Actual {sum_result[0, 0]}')

# Verification for another element (e.g., first row, last column)
# sum_result[0, 2] should be array1[0, 2] + array2[2]
expected_value_0_2 = array1[0, 2] + array2[2]
print(f'Verifying sum_result[0, 2]: Expected {expected_value_0_2}, Actual {sum_result[0, 2]}')

This is what i aksed gemini to generate as my focus was lost and i wanted to confirm some things

In [ ]:
# 4. Pairwise squared distances between 100 points. Loop vs broadcasting with (X - X.reshape(100,1,2)). What shape?

First, let's create our 100 points, each with 2 coordinates. We'll use `np.random.rand` for demonstration purposes.

In [ ]:
import numpy as np

# Create 100 points, each with 2 coordinates (e.g., X, Y)
X = np.random.rand(100, 2)

print("Shape of X:", X.shape)


Now, let's use the broadcasting trick to get all pairwise differences. The hint `(X - X.reshape(100,1,2))` is crucial here. Let's break down what's happening:

*   `X` has shape `(100, 2)`.
*   `X.reshape(100, 1, 2)` changes `X` into an array of shape `(100, 1, 2)`. This means we have 100 'rows', each containing a single 'point' (with 2 coordinates).

When we subtract `X` (effectively broadcast to `(100, 100, 2)`) from `X.reshape(100, 1, 2)` (effectively broadcast to `(100, 100, 2)`), the result `diffs` will have a shape of `(100, 100, 2)`.

Each element `diffs[i, j, :]` will represent the vector difference `X[i, :] - X[j, :]`.


In [ ]:
# Calculate pairwise differences
# X has shape (100, 2)
# X.reshape(100, 1, 2) has shape (100, 1, 2)
# When subtracted, X is broadcast from (100, 2) to (100, 100, 2)
# and X.reshape(100, 1, 2) is broadcast from (100, 1, 2) to (100, 100, 2)
diffs = X - X.reshape(100, 1, 2)
print("Shape of pairwise differences (diffs):", diffs.shape)
print("Example: Difference between point 0 and point 1 (first 5 values):\n", diffs[0, 1, :])
print("Expected: X[0] - X[1]:\n", X[0] - X[1])

Finally, to get the squared Euclidean distance, we square these differences and sum them along the last axis (the coordinate axis).

In [ ]:
# Square the differences
squared_diffs = diffs**2

# Sum along the last axis (axis=2) to get the squared distance
pairwise_squared_distances = np.sum(squared_diffs, axis=2)

print("Shape of pairwise squared distances:", pairwise_squared_distances.shape)
print("First 5x5 block of pairwise squared distances:\n", pairwise_squared_distances[:5, :5])

# As a check, the diagonal elements should be 0 (distance of a point to itself)
print("Diagonal elements (should be close to 0):\n", np.diag(pairwise_squared_distances[:5, :5]))

Found it difficult at this moment, i will study how pairwise works and come back

Finding it hard to do intuitively understand pairwise distance

In [ ]:
# Trap 5: Normalise matrix (100,50) row-wise. Norms have shape (100,). Does data / norms work? What reshape makes it work?

# for normalisation we first calculate the noorm of the array and then divide the whole array with the norm.

array = np.random.rand(100,50)
print(array.shape)

norm_array = np.linalg.norm(array,axis =1) # Row wise!
print(norm_array.shape)

# the reshape that makes data / norm work is reshape(100,1)
# as from rightmost: 1 will be strecthed to 50 and 100 is compatible to 100 only

# 1. I will try without reshaping
try:
  normalised_array = array / norm_array
except ValueError:
  print("Can't broadcast 100,5 wih 100, as for 100, numpy tries to make it a match with 100,5 but it can't that's why: reshape to a broadcastable shape")

# 2. let us reshape it to predicted 100,1 reshaped array so it will work then
try:
  norm_array = norm_array.reshape((100,1))
  normalised_array = array / norm_array
  print("Normalisation compatible with 100,50 and 100,1 array:", normalised_array[0:5,0:5])
except ValueError:
  print("Can't broadcast reshape to a broadcastable shape")

What I predicted: After learning and understanding my lessons from trap 2 and 3: I correctly predicted that (100,) will be changed to (100,1)such that (100,50) and (100, 1) would match and make (100,50) i.e the 1 gets stretched to 50.
just to be sure i used try-except to catch the ValueError object wherever it was raised.

What really Happened,the correct version:(100, 50)
(100,)
Can't broadcast 100,5 wih 100, as for 100, numpy tries to make it a match with 100,5 but it can't that's why: reshape to a broadcastable shape
Normalisation compatible with 100,50 and 100,1 array

How would this silently corrupt the model: Mismatch in broadcasting operands lead to typeError it is better to avoid it or catch it if it ever occurs.

Conclusion: I understood Broadcasting and it's rules also predicted some correct and some codes incorrectly: I am learning and improving day by day:
1st: At reading codes and 2nd: At predicting errors.

Matirx vs Element wise Multiplication

There are two types of multiplications Numpy offers to us users:

1. the `*` multiplication, it is the arthmetic operation that we perform between two arrays. When we use `*` we are multipling the both arrays element by element. The ith element of array1 gets multiplied with ith element of array2. Just like `+` `-` and `/` it works element wise.

2. the `@` operator, my personal favourite, this is a Matrix Multiplication operator, it multiplies two arrays as it is multiplying two matrices. A matrix is not just a block that has number stored up but it is a `TRANSFORMATION` or a function that wraps the space itself. the `@` produces the same result as that of `np.dot(array1,array2)`

In [ ]:
# Code implementing: * and @

Universal Functions Applications: Numpy provided Mathematical functions which are called universal functions or `ufuncs`

predict Shapes and Types of the data with gemini made drills
figure out common traps in broadcasting